# Tiny System 2 — train from scratch on Colab
This notebook calls the same Python package as the CLI and local UI. The move-policy FP32 inference weights are about 32.10 MB. Training checkpoints and GPU memory are additional. A short pilot proves execution, not reasoning quality.

First commit and push `reasoning_harness/` to GitHub, then select **Runtime → Change runtime type → GPU**. GPU allocation is controlled by Colab. You can also run this notebook locally against an existing checkout.

In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY = "https://github.com/Kedhareswer/ML-AI.git"
REF = "main"  # Set to the branch containing the harness, or a published tag.
ROOT = Path("/content/ML-AI") if Path("/content").exists() else Path.cwd()
# Locally, set ROOT to your existing ML-AI checkout.
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REF, "--single-branch", REPOSITORY, str(ROOT)], check=True)
PACKAGE = ROOT / "reasoning_harness"
if not (PACKAGE / "pyproject.toml").exists():
    raise RuntimeError("Set ROOT to your checkout, or commit and push the harness to GitHub first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PACKAGE)], check=True)


## Select durable storage and check compute
Use Google Drive if checkpoints must survive runtime deletion. No Drive credentials belong in this repository. Keep full-size training on a GPU; choose `TINY=True` explicitly for a CPU smoke test.

In [ ]:
import torch
from sys2.policy import PolicyConfig, MovePolicy, train_policy
from sys2.engine import train, load, evaluate, solve
from sys2.tasks import Maze
USE_DRIVE = False
TINY = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE = Path("/content/drive/MyDrive/tiny-system2")
else:
    STORAGE = ROOT / "reasoning_harness" / "runs"
STORAGE.mkdir(parents=True, exist_ok=True)
RUN = STORAGE / "colab-policy-v1"  # Choose a fresh name for each independent experiment.
torch.set_num_threads(2)
cfg = PolicyConfig(width=32, layers=2) if TINY else PolicyConfig()
print(MovePolicy(cfg).info())
print("CUDA available:", torch.cuda.is_available())
if not TINY and not torch.cuda.is_available():
    print("No GPU: the full policy can also train on CPU. Keep checkpoints in durable storage.")


## Run a pilot
The release recipe is 1500 steps with batch size 64. Use a short pilot first if hardware is unfamiliar. Review GPU memory, runtime and held-out metrics before scaling. Use `RESUME=True` after an interruption, and set `TOTAL_STEPS` above the checkpoint step. Resume preserves the original data/model/optimizer settings.

In [ ]:
RESUME = False
TOTAL_STEPS = 1500
summary = train_policy(str(RUN), steps=TOTAL_STEPS, batch_size=4 if TINY else 64,
                       mazes=8 if TINY else 3000, width=cfg.width, layers=cfg.layers,
                       seed=314, resume=RESUME)
print(summary)


## Held-out evaluation
Retry success and one-attempt success share the same first candidate. Every test maze is checked against the training manifest. Low accuracy after a pilot is expected; these metrics determine the next experiment.

In [ ]:
import json
model = load(RUN)
report = evaluate(model, RUN, count=30, size=4, attempts=4)
OUTPUT = RUN / f"eval-step-{TOTAL_STEPS}.json"
if OUTPUT.exists():
    raise FileExistsError("Choose a fresh evaluation filename")
OUTPUT.write_text(json.dumps(report, indent=2))
print({k:v for k,v in report.items() if k != "records"})


In [ ]:
maze = Maze(("S...", ".X..", "..X.", "...G"))
print("\n".join(maze.rows))
print(json.dumps(solve(model, maze, attempts=4), indent=2))

print("Explicit guided search:", json.dumps(model.search(maze, budget=64), indent=2))


## Full benchmark and acceptance gates
A completed run is not an accepted model. This evaluates three seeds, held-out and larger mazes, no-feedback and untrained controls, exact search, confidence intervals, stress cases, answer loss, latency, and memory. Inspect every failed or unrun gate. GPU inference here does not validate independent training runs or a resume after a real Colab disconnect.


In [ ]:
from sys2.benchmarks import benchmark
BENCHMARK_DIR = RUN / f"benchmark-step-{TOTAL_STEPS}"
report = benchmark(RUN, BENCHMARK_DIR, count=10 if TINY else 100,
                   sizes=(4, 5, 6), seeds=(1000000, 2000000, 3000000))
print("LOCAL HYBRID SOLVER ACCEPTED:", report["local_hybrid_solver_ready"])
print("Original neural feedback research gates:", report["ready"])
print(json.dumps(report["gates"], indent=2))
print("Full evidence:", BENCHMARK_DIR)


## Scale only after inspection
For a longer run, set `RESUME=True`, increase `TOTAL_STEPS` above the saved step, and rerun the training cell. The tested policy uses FP32; CUDA execution remains to be checked on your allocated runtime. Resume retains the pilot batch size; start a separate run to change optimizer/data settings. Also evaluate `size=5` to check generalization. Save the complete run directory to retain resumability; inference alone needs `weights.pt` and `config.json`.

Local interactive UI: `sys2 ui --run PATH_TO_RUN --port 7860`. Notebook inference above works directly in Colab without a public tunnel. GPU mixed precision and the Colab runtime must be validated on the allocated machine; local development only had a CPU.